# Prepare the contend based on one model

This notebook contains the steps to generate the SSOT from an ODM model and render output channels.

This notebook will be executed by the run.bat

In [1]:
odm_source_folder = 'IM'
destination_folder = 'content'

# Skip generation of the SSOT from ODM? 
skip_ssot_generation = False

# Location of the python tools
tools_path = 'pythonWork/pythonSource'

In [2]:
notebook_version = "0.6"

In [3]:
import sys
import os
from pathlib import Path
import glob
import json

In [4]:
import logging
from logging import handlers

os.makedirs('log', exist_ok=True)
logfile = 'log/generator.log'

handler = handlers.RotatingFileHandler(logfile, maxBytes=(1024*1024*10), backupCount=10)
handler.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

console_log_handler = logging.StreamHandler()
console_formatter = logging.Formatter("%(levelname)s - %(message)s")
console_log_handler.setFormatter(console_formatter)
console_log_handler.setLevel(logging.INFO)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logger.addHandler(handler)
logger.addHandler(console_log_handler)

In [5]:
sys.argv[0]

'/opt/homebrew/anaconda3/lib/python3.9/site-packages/ipykernel_launcher.py'

In [6]:
import argparse

parser = argparse.ArgumentParser(description='Generate SSOT and diagrams from ODM model')
parser.add_argument('--model', dest='source_folder', default=odm_source_folder)
parser.add_argument('--destination', dest='destination_folder', default=destination_folder)
parser.add_argument('--skip-ssot', '-f', action='store_true', dest='skip_odm', help="Skip generation of SSOT out of Oracle Database Modeler model")
parser.add_argument('--skip-web', '-w', action='store_true', dest='skip_web', help="Skip generation of static web content")
parser.add_argument('--confluence', '-c', action='store_true', dest='confluence', help="Skip generation of Confluence content")
parser.add_argument('--sharepoint', '-s', action='store_true', dest='sharepoint', help="Enable generation of Sharepoint content")
parser.add_argument('--version', action='store_true')

arguments = argparse.Namespace()

if len(sys.argv) > 0 and '.py' in sys.argv[0] and not 'ipykernel' in sys.argv[0]:
    arguments = parser.parse_args()
    odm_source_folder = arguments.source_folder
    destination_folder = arguments.destination_folder
    skip_ssot_generation = arguments.skip_odm
    if arguments.version:
        print(f"Version {notebook_version}")
        exit(0)
else:
    # in jupyter environment
    odm_source_folder = '/Users/bue/dev/fyyccim-refmodels/riddle/IM'
    destination_folder = 'content'
    tools_path = os.path.abspath('../../pythonWork/pythonSource')
    skip_ssot_generation = False


In [7]:
logger.info(f"Starting generator version {notebook_version}")

INFO - Starting generator version 0.6


### Safeguards


In [8]:
models = glob.glob(odm_source_folder + '/*.[dD][mM][dD]')
if len(models) < 1:
    logger.fatal(f"No model file (*.dmd) found in source folder {os.path.abspath(odm_source_folder)}.")
    exit(2)

model = models[0]
if len(models) > 1:
    for name in models:
        # pick model with shortest name
        if len(name) < len(model):
            model = name
    logger.warning(f"Found {len(models)} models in {os.path.abspath(odm_source_folder)} using {model}")
    # disable other models?
    for name in models:
        if name != model:
            os.rename(name, os.path.splitext(name)[0] + '.hidden')

In [9]:
base_path = Path(odm_source_folder)
assert os.path.isdir(odm_source_folder), "Cannot find source folder {}".format(odm_source_folder)
project_files = list(base_path.glob('*.dmd'))
assert len(project_files) == 1, "Cannot find exactly 1 ODM .dmd file in source folder {}: {}".format(odm_source_folder, project_files)
logger.info(f"Processing the information model in {os.path.abspath(base_path)}")
from IPython.core.display import HTML
HTML('<span style="font-family: Impact; font-size:48px">Processing the information model in<br/><span style="color: darkorange">{0}</span></span>'.format(os.path.abspath(odm_source_folder)))

INFO - Processing the information model in /Users/bue/dev/fyyccim-refmodels/riddle/IM


In [10]:
if not (os.path.exists(tools_path)
        and os.path.isfile(os.path.join(tools_path, 'IM_ODM', 'transferModel.py'))):
    logging.fatal(f"Tools not in expected path {tools_path}")
    exit(3)

logger.debug(f"Working with tools in {tools_path}")

# Add toolbox to python library path
sys.path.insert(0, os.path.abspath(tools_path))

# HACK around issue #xxx
sys.path.insert(1, os.path.abspath(os.path.join(tools_path, 'IM_db')))
sys.path.insert(1, os.path.abspath(os.path.join(tools_path, 'IM_WEB')))

from SSOT_infra import parameters
from IM_db.IM_JSON import JSModel
from IM_WEB.IM_HTML import entityenviron
from IM_WEB.IM_HTML import drawiodiagram

In [11]:
from SSOT_infra import logmessages

def tap_logmessages(message: str):
    logger.warning(message)

logmessages.logtap = tap_logmessages

# Process the datasource

In [12]:
global parameter

parameters.initparam(odm_source_folder)

config_folder_before = None

odm_config_folder = os.path.join(odm_source_folder, parameters.odmKonfDirec())
if not os.path.isdir(odm_config_folder):
#    parameters.parameter['odmkonfdirec'] = 'Configuration/'
    odm_config_folder = os.path.join(odm_source_folder, parameters.odmKonfDirec())
    logger.warning(f"Patching config folder to {parameters.odmKonfDirec()}")
    config_folder_before = os.path.join(odm_source_folder, 'Configuration')
    assert os.path.isdir(config_folder_before), f"Missing configuration folder {config_folder_before}"
    os.rename(config_folder_before, odm_config_folder)

assert os.path.isdir(odm_config_folder), f"Configuration folder {os.path.abspath(odm_config_folder)} not found"
assert parameters.odmKonfDirec()[-1] == '/'

In [13]:
sqlfilepath = os.path.join(tools_path, 'IM_db/sqlfiles/')
#parameters.parameter['sqlpath'] = sqlfilepath
#assert os.path.isfile(parameters.sqlfilepath()), f"Schema not found {parameters.sqlfilepath()}"

In [14]:
from IM_ODM import fillDB

if not skip_ssot_generation:
    logger.info('Updating database {db} from model {odm}'.format(db=parameters.dbFilePath(), odm=parameters.dbDirect()))
    try:
        fillDB.filldbmain(odm_source_folder, createnewdb=True)
        logger.info(f"Sucessfully updated {parameters.dbFilePath()}")
    except:
        print('Consult logfile {}'.format(parameters.logfilepath()))
        raise

INFO - Updating database /Users/bue/dev/fyyccim-refmodels/riddle/DB/riddle.db from model /Users/bue/dev/fyyccim-refmodels/riddle/DB/
WARNING - in Relation D5C64CE3-B262-DA6D-0371-7B051B77E9D7: Entity Id D106E4B9-DF50-9887-E8C4-5B81F389229E or 84E56FAB-251C-8920-49FA-0BE7F1C75905 not found. Datenleichen von Relation mit gelöschten Entities
WARNING - in Relation AF0F054F-813D-0B0C-C168-F52751653AD9: Entity Id E1455130-5348-4334-7A73-BBDB38EEF58F or F31B42E9-659F-0C7B-4215-AFE87CF401D4 not found. Datenleichen von Relation mit gelöschten Entities
WARNING - in Relation 0902E968-F67A-9E88-4844-CC8F534F055E: Entity Id 07BDF8BC-95D9-2C86-6F4E-BE8D0881F9D4 or 8D3423A8-D897-B843-6E7A-1B727D8E0F91 not found. Datenleichen von Relation mit gelöschten Entities
WARNING - in Relation 64B5572D-FDAE-6AF8-D322-0C2E85AB51A3: Entity Id D572E71A-C2A1-F402-99D5-A45619634DD4 or D572E71A-C2A1-F402-99D5-A45619634DD4 not found. Datenleichen von Relation mit gelöschten Entities
WARNING - in Relation 9195BF1C-248C

/opt/homebrew/anaconda3/lib/python3.9/site-packages/ipykernel_launcher.py:
  => database /Users/bue/dev/fyyccim-refmodels/riddle/DB/riddle.db version 1.5 for model riddle created


WARNING - in Relation C4748029-3C2E-8628-3141-8177EA438318: Entity Id 07BDF8BC-95D9-2C86-6F4E-BE8D0881F9D4 or FA2D0D88-3F19-9D4C-4834-CD2D968DBEFA not found. Datenleichen von Relation mit gelöschten Entities
WARNING - in Relation D9ED4DD1-4243-126E-AF64-48CC57215AFC: Entity Id 3A41D47C-420D-27D6-54F9-7D0D165F2F29 or 9D0668B7-80FA-AAFC-796F-2E2AA388E8EE not found. Datenleichen von Relation mit gelöschten Entities
WARNING - in Relation 91799BAE-4EF0-754A-9192-F83698E36018: Entity Id 3A41D47C-420D-27D6-54F9-7D0D165F2F29 or E1455130-5348-4334-7A73-BBDB38EEF58F not found. Datenleichen von Relation mit gelöschten Entities
WARNING - in Relation D031054C-4B81-C0C9-FD8B-CAB267C207F6: Entity Id 07BDF8BC-95D9-2C86-6F4E-BE8D0881F9D4 or 8D3423A8-D897-B843-6E7A-1B727D8E0F91 not found. Datenleichen von Relation mit gelöschten Entities
WARNING - in Relation ADC48189-147A-BAA5-3D7C-0619CA8A8531: Entity Id 5E4F3063-A75F-F7CC-41DE-B566A2EE99D7 or E1455130-5348-4334-7A73-BBDB38EEF58F not found. Datenleich

In [15]:
database_file = os.path.abspath(parameters.dbFilePath())
json_file = os.path.splitext(database_file)[0] + '.json'
assert os.path.isfile(json_file), f"SSOT file {json_file} not found"
logger.info(f"Loading SSOT from {json_file}")

INFO - Loading SSOT from /Users/bue/dev/fyyccim-refmodels/riddle/DB/riddle.json


In [16]:
with open(json_file) as f:
    data = json.load(f)
assert len(data) > 0, f'Config is empty :-(' 

In [17]:
jsmodel = JSModel.readfromfile(pfilename=json_file)

In [18]:
import gettext

class Translator:
    """Translate strings"""
    logger = logging.getLogger("Translator")
    
    def __init__(self, language: str):
        self.language = language
        self.translator = gettext.translation('confluence-publisher', './locale', fallback=True, languages=[language])
        self.title_format = '{title} - [{language}]'
        self.logger = logging.getLogger("Translator " + language)
        
    def tr(self, element) -> str:
        if isinstance(element, dict):
            """If the value provided is a field containing translations, use them"""
            text = element.get(self.language)
            if text is None: #and len(element.values()) > 0:
                result = list(element.values())[0]
                if result is not None:
                    self.logger.warning('No translation for {text}. Falling back to {} from {}'.format(text, str(element)))
                    text = result
            if not text:
                return ''
            # Strip leading translation marker
            #if '*de* ' in text:
            #    text = text.replace('*de* ', '', 1)
            return text

        # Fallback to gettext if not a dict
        if isinstance(element, str):
            translated = self.translator.gettext(element)
            return translated

        self.logger.warning("Cannot translate element '{0}' of type {1}".format(element, type(element)))
        return None
    
    def gettext(self, text: str):
        result = self.translator.gettext(text)
        if result == text:
            self.logger.warning("No translation for {0}".format(text))
        return result
    
    def translator(self):
        return self.translator
    
    def title_language(self, title: str) -> str:
        """Create unique confluence page title per translation"""
        return self.title_format.format(title = title, language = self.language)
    
    def key_lang(self, key: str) -> str:
        return key + '-' + self.language
    
    def lang(self) -> str:
        return self.language

In [19]:
translators = { language: Translator(language) for language in data['languages'] }
translators

{'en': <__main__.Translator at 0x7fb4889374f0>}

In [20]:
def sanitize_filename(name: str) -> str:
    return "".join(c for c in name if c.isalnum() or c in ('.', '-', '_', ' ')).rstrip()

# Render draw.io diagrams

In [21]:
from lxml import etree

from tqdm.autonotebook import tqdm

content_root = '.'

diagram_count = len(data['diagrams'].keys())*len(translators)

path = os.path.join(destination_folder, 'diagrams')            
folder = os.path.join(content_root, path)
os.makedirs(folder, exist_ok=True)
logger.info(f"Rendering diagrams to {os.path.abspath(folder)}")

with tqdm(total=diagram_count, dynamic_ncols=True, unit='Diagram') as pbar:
    for lang, translator in translators.items():
        for key, diagram in data['diagrams'].items():

            filename = f"{key}-{sanitize_filename(diagram['name'])}-{lang}.drawio"
            file = os.path.join(folder, filename)
            
            pbar.set_description(f"Generating diagram {key} '{diagram['name']}' [{lang}] to {file}")
            draw_io_xml = drawiodiagram.create_diagram(key, jsmodel, translator)
            with open(file, 'wb') as out:
                out.write(etree.tostring(draw_io_xml))
                logger.debug(f"Diagram '{diagram['name']}' stored in draw.io format to {filename}")
            pbar.update(1)

<ipython-input-21-745a61821c58>:3: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm
INFO - Rendering diagrams to /Users/bue/dev/fyyccim-tools/notebooks/mig/content/diagrams


  0%|          | 0/3 [00:00<?, ?Diagram/s]

In [22]:
if config_folder_before is not None:
    logger.warning(f"Moving configuration folder back to {config_folder_before}")
    os.rename(odm_config_folder, config_folder_before)

In [23]:
logger.info(f"Successfully created {diagram_count} diagrams to {os.path.join(destination_folder, 'diagrams')}")

INFO - Successfully created 3 diagrams to content/diagrams


# Create web content

In [24]:
from IM_HTML import printHTML
import listWebdoku
from IM_OBJECTS import Languagetext

printHTML.setmodel(jsmodel)
printHTML.setWebDirec(p_webdirec=None)

listWebdoku.listwebmain(plang=Languagetext.reportLang())

create web-files for language en in file /Users/bue/dev/fyyccim-refmodels/riddle/Web/riddle.html
create web-files for system Example System A in file /Users/bue/dev/fyyccim-refmodels/riddle/Web/Example System A.html
